In [1]:
import bs4
import requests
import json
import re
import openpyxl #import load_workbook, Workbook, create_sheet
import os
import pandas as pd
import copy
#from googletrans import Translator       
#영어 발음 참고: https://github.com/mphilli/English-to-IPA

def crawl(cate1, cate2, url, paint, brand):  
    html = requests.get(url)
    soup = bs4.BeautifulSoup(html.text, 'lxml')
    
    title = soup.find('div',class_='layout__item tools-heading')
    PAINT_NAME = title.find('h2').string
    
    metadata = soup.find('div',class_='filter-grid')
    links = [tag.get('href') for tag in soup.find_all('a', class_='layout__item')[:-2]]
    for link in links:

        html = requests.get(link)
        soup = bs4.BeautifulSoup(html.text, 'lxml')
        meta = soup.select_one('section div.wrap>div:nth-of-type(3)')

        divs = [child for child in meta.children if isinstance(child, bs4.element.Tag)]
        paint['IMG1'].append(divs[0].find('img').get('src'))
        paint['IMG2'].append(divs[1].find('img').get('src'))

        specs = divs[1].find('div', class_='layout__item colour-details').find_all('div', class_='layout__item colour__detail')
        
        paint['PAINT_NAME'].append(PAINT_NAME)
        paint['PAINT_NAME_KOR'].append(None)
        paint['COLOR'].append(specs[0].contents[1].replace(': ',''))
        paint['COLOR_KOR'].append(None)
        paint['BRAND'].append(brand)
        paint['BRAND_KOR'].append(None)
        paint['SERIES'].append(specs[2].contents[1].replace(': ',''))
        if 'ml' in specs[4].contents[1]:
            paint['UNIT'].append('ml')
        elif 'g' in specs[4].contents[1]:
            paint['UNIT'].append('g')
        else:
            paint['UNIT'].append(None)
        paint['SIZE'].append(re.sub(r'\s|:|in|and|tube|tin|ml|g','',specs[4].contents[1])) #.split(',')
        paint['PIG'].append(specs[5].contents[1].replace(': ','')) #.split('-|,') ex)PB 73, PV 23, PV 19
        
        #내광성
        lf = specs[7].contents[1]
        if '1' in lf or '2' in lf:
            paint['LIGHTFASTNESS'].append('High')
            paint['LIGHTFASTNESS_KOR'].append('높음')
        elif '3' in lf:
            paint['LIGHTFASTNESS'].append('Medium')
            paint['LIGHTFASTNESS_KOR'].append('보통')
        elif '4' in lf or '5' in lf:
            paint['LIGHTFASTNESS'].append('Low')
            paint['LIGHTFASTNESS_KOR'].append('낮음')
        else:
            paint['LIGHTFASTNESS'].append(None)
            paint['LIGHTFASTNESS_KOR'].append(None)
            
            
        #투명도
        tp = specs[8].contents[1].replace(': ','')
        if 'Semi' in tp:
            paint['TRANSPARENCY'].append('Semi-Transparent')
            paint['TRANSPARENCY_KOR'].append('반투명')
        elif 'Transparent'==tp:
            paint['TRANSPARENCY'].append('Transparent')
            paint['TRANSPARENCY_KOR'].append('투명')
        elif 'Opacue'==tp:
            paint['TRANSPARENCY'].append('Opacue')
            paint['TRANSPARENCY_KOR'].append('불투명')
        else:
            paint['TRANSPARENCY'].append(None)
            paint['TRANSPARENCY_KOR'].append(None)
        
        #기타(또 다르게 크롤링해야 하는 것. quality, vegan, description)
        paint['DESCRIPTION'].append(None)
        paint['DESCRIPTION_KOR'].append(None)
        paint['VEGAN'].append(None)
        paint['QUALITY'].append(None)
        paint['QUALITY_KOR'].append(None)
        paint['BINDER'].append(None)
        paint['BINDER_KOR'].append(None)
        paint['TEXTURE'].append(None)
        paint['TEXTURE_KOR'].append(None)
        paint['COLOR_GROUP'].append(None)
        paint['COLOR_GROUP_KOR'].append(None)
    
    return paint

In [2]:
def open_csv(paint_template, urls, brand):
    for path, cate, url in urls:
        print(path)
        cate1 = path.split('/')[-1]
        cate2 = cate+' '+cate1
        paint = copy.deepcopy(paint_template)
        
        dic = crawl(cate1, cate2, url, paint, brand)
        
        if not os.path.isdir(path):
            os.mkdir(path)
        fn = path+'/'+cate+'.xlsx'
        writer = pd.ExcelWriter(fn, engine='xlsxwriter')
        pd_df = pd.DataFrame(dic)
        pd_df.to_excel(writer, sheet_name=brand)
        writer.save()
        

In [3]:

paint = {
    'PAINT_NAME':[],
    'PAINT_NAME_KOR':[],
    'BRAND':[],
    'BRAND_KOR':[],
    'COLOR_GROUP':[],
    'COLOR_GROUP_KOR':[],
    'COLOR':[],
    'COLOR_KOR':[],
    'SERIES':[],
    'TEXTURE':[],
    'TEXTURE_KOR':[],
    'BINDER':[],
    'BINDER_KOR':[],
    'TRANSPARENCY':[],
    'TRANSPARENCY_KOR':[],
    'LIGHTFASTNESS':[],
    'LIGHTFASTNESS_KOR':[],
    'QUALITY':[],
    'QUALITY_KOR':[],
    'PIG':[],
    'SIZE':[],
    'UNIT':[],
    'VEGAN':[],
    'DESCRIPTION':[],
    'DESCRIPTION_KOR':[],
    'IMG1':[],
    'IMG2':[]
}

brand = 'Old Holland'

urls = [('../paints/Oil','Classic','https://www.oldholland.com/oil-colours/'),
    ('../paints/Acrylic','Classic','https://www.oldholland.com/acrylics/'),
    ('../paints/Watercolor','Classic','https://www.oldholland.com/watercolours/')]

open_csv(paint,urls, brand)

../paints/Oil
../paints/Acrylic
../paints/Watercolor
